In [4]:
# Cell 1: Check and load dataset

import pandas as pd
import os

print("Files in current directory:")
print(os.listdir("."))

# Load dataset
df = pd.read_csv("fake_job_postings.csv")

print("\nDataset shape:", df.shape)
print("\nFirst 3 rows:")
df.head(3)

Files in current directory:
['.venv', 'fakejobs_traning.ipynb', 'fake_job_postings.csv']

Dataset shape: (17880, 18)

First 3 rows:


,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0


In [6]:
# Diagnostic: check column names

import pandas as pd

df = pd.read_csv("fake_job_postings.csv")

print("Column names in the dataset:")
print(df.columns.tolist())

print("\nFirst 2 rows:")
df.head(2)

Column names in the dataset:
['job_id', 'title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'telecommuting', 'has_company_logo', 'has_questions', 'employment_type', 'required_experience', 'required_education', 'industry', 'function', 'fraudulent']

First 2 rows:


,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0


In [7]:
# Cell 2: Preprocessing and train-test split (fixed column name)

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder

# Re-load dataset
df = pd.read_csv("fake_job_postings.csv")

# -------------------------------
# 1. Combine text fields
# -------------------------------
df["text"] = (
    df["title"].fillna("") + " " +
    df["description"].fillna("") + " " +
    df["requirements"].fillna("")
)

# Target (note lowercase 'fraudulent')
y = df["fraudulent"].values

# Categorical columns to use
cat_cols = [
    "employment_type",
    "required_experience",
    "required_education",
    "industry",
    "function",
    "telecommuting",
    "has_company_logo",
    "has_questions"
]

# -------------------------------
# 2. Basic text cleaning
# -------------------------------
def clean_text(t):
    t = str(t).lower()
    t = " ".join(t.split())  # remove extra spaces
    return t

df["text_clean"] = df["text"].apply(clean_text)

# -------------------------------
# 3. TF-IDF on text
# -------------------------------
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=5,
    stop_words="english"
)

X_tfidf = tfidf.fit_transform(df["text_clean"])

# -------------------------------
# 4. One-hot encode categorical features
# -------------------------------
X_cat_df = df[cat_cols].fillna("Unknown")
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_cat = ohe.fit_transform(X_cat_df)

# -------------------------------
# 5. Combine features
# -------------------------------
X = np.hstack([X_tfidf.toarray(), X_cat])

# -------------------------------
# 6. Train-test split
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Training set fake %:", y_train.sum() / len(y_train) * 100)

X_train shape: (14304, 5204)
X_test shape: (3576, 5204)
Training set fake %: 4.844798657718121


In [8]:
# Cell 3: Train and evaluate models

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# (X_train, X_test, y_train, y_test are from Cell 2)

# -------------------------------
# 1. Train Logistic Regression
# -------------------------------
lr = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print("=== Logistic Regression ===")
print(classification_report(y_test, y_pred_lr, digits=4, target_names=["Real", "Fake"]))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_lr))

# -------------------------------
# 2. Train Random Forest
# -------------------------------
rf = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("\n=== Random Forest ===")
print(classification_report(y_test, y_pred_rf, digits=4, target_names=["Real", "Fake"]))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_rf))

# -------------------------------
# 3. Choose best model (by F1 on fake class)
# -------------------------------
f1_lr_fake = f1_score(y_test, y_pred_lr, pos_label=1)
f1_rf_fake = f1_score(y_test, y_pred_rf, pos_label=1)

print("\nF1 score for FAKE class:")
print("Logistic Regression:", f1_lr_fake)
print("Random Forest:", f1_rf_fake)

if f1_rf_fake >= f1_lr_fake:
    best_model = rf
    best_name = "RandomForest"
else:
    best_model = lr
    best_name = "LogisticRegression"

print("\nBest model chosen:", best_name)

=== Logistic Regression ===
              precision    recall  f1-score   support

        Real     0.9936    0.9641    0.9787      3403
        Fake     0.5547    0.8786    0.6801       173

    accuracy                         0.9600      3576
   macro avg     0.7742    0.9214    0.8294      3576
weighted avg     0.9724    0.9600    0.9642      3576

Confusion matrix:
[[3281  122]
 [  21  152]]

=== Random Forest ===
              precision    recall  f1-score   support

        Real     0.9829    0.9971    0.9899      3403
        Fake     0.9194    0.6590    0.7677       173

    accuracy                         0.9807      3576
   macro avg     0.9511    0.8280    0.8788      3576
weighted avg     0.9798    0.9807    0.9792      3576

Confusion matrix:
[[3393   10]
 [  59  114]]

F1 score for FAKE class:
Logistic Regression: 0.680089485458613
Random Forest: 0.7676767676767676

Best model chosen: RandomForest


In [9]:
# Cell 4: Save supervised model and build clustering pipeline

import joblib
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# -------------------------------
# 1. Save supervised model + preprocessors
# -------------------------------
joblib.dump({
    "model": best_model,
    "tfidf": tfidf,
    "ohe": ohe,
    "cat_cols": cat_cols,
    "best_name": best_name
}, "best_fake_job_model.pkl")

print("Saved best_fake_job_model.pkl")

# -------------------------------
# 2. Clustering pipeline (unsupervised)
# -------------------------------

# Re-load dataset to be safe
df = pd.read_csv("fake_job_postings.csv")

# Use description text for clustering
df["desc_text"] = df["description"].fillna("").apply(lambda t: " ".join(str(t).lower().split()))

tfidf_cluster = TfidfVectorizer(
    max_features=2000,
    ngram_range=(1, 2),
    min_df=10,
    stop_words="english"
)
X_desc = tfidf_cluster.fit_transform(df["desc_text"])

# Dimensionality reduction
svd = TruncatedSVD(n_components=50, random_state=42)
X_reduced = svd.fit_transform(X_desc)

# Choose K using silhouette score
sil_scores = []
Ks = range(3, 9)
for k in Ks:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init="auto")
    labels = kmeans.fit_predict(X_reduced)
    sil_scores.append(silhouette_score(X_reduced, labels))

best_k = Ks[np.argmax(sil_scores)]
print("\nBest number of clusters (by silhouette):", best_k)

# Final clustering
kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init="auto")
clusters = kmeans_final.fit_predict(X_reduced)
df["cluster"] = clusters

# Save clustering pipeline
joblib.dump({
    "tfidf_cluster": tfidf_cluster,
    "svd": svd,
    "kmeans": kmeans_final,
    "best_k": best_k
}, "job_clustering_pipeline.pkl")

print("Saved job_clustering_pipeline.pkl")

# Save cluster assignments
if "job_id" in df.columns:
    df[["job_id", "cluster"]].to_csv("job_clusters.csv", index=False)
else:
    df[["cluster"]].to_csv("job_clusters.csv", index=False)

print("Saved job_clusters.csv")
print("Cell 4 complete.")

Saved best_fake_job_model.pkl

Best number of clusters (by silhouette): 3
Saved job_clustering_pipeline.pkl
Saved job_clusters.csv
Cell 4 complete.


In [11]:
# Cell 5: Save model files to current folder (local run)

import joblib
import pandas as pd
import os

print("Current working directory:", os.getcwd())
print("Files in current directory:")
print(os.listdir("."))

# These objects already exist from previous cells:
# - best_model, tfidf, ohe, cat_cols, best_name
# - tfidf_cluster, svd, kmeans_final, best_k
# - df with "cluster" column

# 1. Save supervised model + preprocessors
joblib.dump({
    "model": best_model,
    "tfidf": tfidf,
    "ohe": ohe,
    "cat_cols": cat_cols,
    "best_name": best_name
}, "best_fake_job_model.pkl")

print("Saved best_fake_job_model.pkl")

# 2. Save clustering pipeline
joblib.dump({
    "tfidf_cluster": tfidf_cluster,
    "svd": svd,
    "kmeans": kmeans_final,
    "best_k": best_k
}, "job_clustering_pipeline.pkl")

print("Saved job_clustering_pipeline.pkl")

# 3. Save cluster assignments
if "job_id" in df.columns:
    df[["job_id", "cluster"]].to_csv("job_clusters.csv", index=False)
else:
    df[["cluster"]].to_csv("job_clusters.csv", index=False)

print("Saved job_clusters.csv")
print("All model files saved in current folder.")

Current working directory: c:\Users\Harshada Kawale\OneDrive\ml clg\ML model
Files in current directory:
['.venv', 'best_fake_job_model.pkl', 'fakejobs_traning.ipynb', 'fake_job_postings.csv', 'job_clustering_pipeline.pkl', 'job_clusters.csv']
Saved best_fake_job_model.pkl
Saved job_clustering_pipeline.pkl
Saved job_clusters.csv
All model files saved in current folder.
